# Dataset 2 — Credit Card Fraud Detection 2023 (Google Colab)

Self-contained Colab port of the Dataset 2 pipeline.

**Pipeline:**
1. Load Kaggle Credit Card Fraud 2023 dataset via `kagglehub`
2. Clean + light EDA (3 plots: class distribution, amount distribution, feature/Class correlation)
3. **10-fold shuffled `KFold` cross-validation** (no SMOTE, no stratification — see note below)
4. **Algorithm 1 — Logistic Regression** (full pipeline + 4 plots)
5. **Algorithm 2 — Random Forest** (full pipeline + 4 plots)
6. **Algorithm 3 — Feed-Forward Neural Network** (full pipeline + 4 plots, GPU-aware)

**Metrics:** Precision, Recall, F1, MCC, AUC-ROC.
**Artefacts:** PNGs + CSVs are written to `./results/` for download.

> **Why no SMOTE and no stratification?** This Kaggle 2023 dataset is **pre-balanced**
> at ~50/50 (~284K legit / ~284K fraud). SMOTE would inject synthetic noise without
> improving class balance, and stratified folds add nothing when both classes are
> already equally represented in any random shuffle. We use plain shuffled `KFold`.

> Tip: switch the Colab runtime to GPU (`Runtime → Change runtime type → GPU`)
> before running the FFNN section.


## Phase 1 — Environment setup

In [ ]:
# Install dependencies. Colab already ships pandas/numpy/sklearn/matplotlib/seaborn/torch,
# so we only need kagglehub for dataset access. imbalanced-learn is NOT required here
# (no SMOTE — dataset is already balanced).
!pip install -q "kagglehub[pandas-datasets]"


### Kaggle authentication

`kagglehub` needs Kaggle API credentials. The easiest path in Colab is to upload your
`kaggle.json` (Account → Create New API Token):

```python
from google.colab import files
files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
```

Or call `kagglehub.login()` interactively.


In [ ]:
# Standard library + scientific stack imports.
# Everything the notebook needs is imported up-front so later cells stay focused on logic.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn: scaling, splitting, models (LR + RF), and metrics.
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, matthews_corrcoef,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
)

# PyTorch for the FFNN. TensorDataset + DataLoader give us mini-batch iteration.
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# kagglehub pulls the dataset straight into a pandas DataFrame.
import kagglehub
from kagglehub import KaggleDatasetAdapter

# ── Shared constants ──
# RANDOM_STATE seeds every stochastic step (KFold shuffle, sklearn models, torch,
# numpy, DataLoader generator) so the notebook is fully reproducible.
RANDOM_STATE = 42
# 10 folds gives 10 independent (train, test) measurements per algorithm — enough
# to compute mean ± std with reasonable confidence on a 568K-row balanced dataset.
N_FOLDS = 10
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Auto-detect GPU. The FFNN trains roughly an order of magnitude faster on CUDA.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {DEVICE}")


def save_fig(fig, name):
    """Save a figure to ./results/ AND display it inline in the notebook.

    Saving to disk lets us download all artefacts as a zip at the end, while
    plt.show() keeps the inline rendering Colab users expect.
    """
    path = os.path.join(RESULTS_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Saved: {path}")
    plt.show()
    plt.close(fig)


## Phase 2 — Load data from Kaggle

In [ ]:
# Pull the latest version of the Kaggle 2023 credit card fraud dataset directly
# into a DataFrame. KaggleDatasetAdapter.PANDAS handles download + CSV parsing.
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "nelgiriyewithana/credit-card-fraud-detection-dataset-2023",
    "creditcard_2023.csv",
)

print("First 5 records:")
print(df.head())


In [ ]:
# Cleaning + sanity checks. This dataset has 31 columns: id, V1–V28, Amount, Class.
print("=" * 60)
print("1. LOADING DATA")
print("=" * 60)

# Drop the 'id' column. It's a sequential row index and carries no predictive
# information — keeping it would let the model memorise row positions instead
# of learning fraud patterns.
if "id" in df.columns:
    df = df.drop(columns=["id"])
    print("  Dropped 'id' column (sequential index, not a real feature)")

# Class arrives as int already in this dataset (unlike Dataset 3 where it's a string).
df["Class"] = df["Class"].astype(int)

print(f"Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")
print(f"\nBasic statistics:\n{df.describe()}")

# Missing-value check — the Kaggle 2023 release is clean, but verify rather than assume.
missing = df.isnull().sum()
total_missing = missing.sum()
print(f"\nMissing values: {total_missing}")
if total_missing > 0:
    print(missing[missing > 0])

# Duplicate-row check — duplicates would leak information across folds.
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates:,}")
if duplicates > 0:
    print("  -> Dropping duplicates")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  -> New shape: {df.shape[0]:,} rows")


## Phase 3 — Exploratory Data Analysis (3 retained plots)

In [ ]:
# ── Plot 1: Class distribution ──
# Unlike Dataset 3 (severe <1% fraud), this plot confirms publisher-applied 50/50
# balance — the central justification for skipping SMOTE later in the pipeline.
print("=" * 60)
print("2. CLASS DISTRIBUTION")
print("=" * 60)

class_counts = df["Class"].value_counts().sort_index()
class_pct = df["Class"].value_counts(normalize=True).sort_index() * 100

print(f"  Legitimate (0): {class_counts[0]:>8,}  ({class_pct[0]:.4f}%)")
print(f"  Fraudulent (1): {class_counts[1]:>8,}  ({class_pct[1]:.4f}%)")
print(f"  Ratio: 1 : {class_counts[0] / class_counts[1]:.2f}  (near-perfect balance)")

colors = ["#2ecc71", "#e74c3c"]
labels = ["Legitimate (0)", "Fraudulent (1)"]

# Side-by-side bar (raw counts) + pie (proportions). No explode on the pie because
# the slices are equal — no minority class to highlight.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(labels, class_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Transaction Class Counts", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pct.values)):
    axes[0].text(i, cnt + cnt * 0.01, f"{cnt:,}\n({pct:.3f}%)", ha="center", fontsize=10)

axes[1].pie(class_counts.values, labels=labels, colors=colors,
            autopct="%1.3f%%", startangle=90)
axes[1].set_title("Class Proportion", fontsize=14, fontweight="bold")

fig.suptitle("Pre-Balanced Class Distribution in Dataset 2 (~50/50)",
             fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "01_class_distribution.png")


In [ ]:
# ── Plot 2: Transaction amount distribution by class ──
# Amount is the only non-PCA feature so it's worth a careful look. We plot the
# full range AND a zoomed view (≤ $500) because a few large outliers stretch
# the x-axis and hide the bulk of the distribution.
print("=" * 60)
print("3. TRANSACTION AMOUNT ANALYSIS")
print("=" * 60)

legit = df[df["Class"] == 0]["Amount"]
fraud = df[df["Class"] == 1]["Amount"]

# Print descriptive stats first — useful for the report write-up.
print(f"  {'Metric':<20} {'Legitimate':>14} {'Fraudulent':>14}")
print(f"  {'-' * 48}")
print(f"  {'Mean':<20} {legit.mean():>14.2f} {fraud.mean():>14.2f}")
print(f"  {'Median':<20} {legit.median():>14.2f} {fraud.median():>14.2f}")
print(f"  {'Std Dev':<20} {legit.std():>14.2f} {fraud.std():>14.2f}")
print(f"  {'Max':<20} {legit.max():>14.2f} {fraud.max():>14.2f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# density=True normalises histograms by area. We want comparable shape, not raw counts.
axes[0].hist(legit, bins=80, alpha=0.6, color="#2ecc71", label="Legitimate", density=True)
axes[0].hist(fraud, bins=80, alpha=0.6, color="#e74c3c", label="Fraudulent", density=True)
axes[0].set_title("Amount Distribution (Full Range)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Amount"); axes[0].set_ylabel("Density"); axes[0].legend()

# Zoomed view — most transactions are < $500, so this is where the discriminative
# information lives.
axes[1].hist(legit[legit <= 500], bins=80, alpha=0.6, color="#2ecc71",
             label="Legitimate", density=True)
axes[1].hist(fraud[fraud <= 500], bins=80, alpha=0.6, color="#e74c3c",
             label="Fraudulent", density=True)
axes[1].set_title("Amount Distribution (≤ $500)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Amount"); axes[1].set_ylabel("Density"); axes[1].legend()

fig.suptitle("Transaction Amount: Legitimate vs Fraudulent",
             fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "02_amount_distribution.png")


In [ ]:
# ── Plot 3: Pearson correlation of each feature with Class ──
# V1–V28 are PCA components, so each has a different (unknown) interpretation.
# Correlation with the Class label tells us which components carry the most
# linear discriminative signal — useful for the LR feature-coefficient plot
# later, and as a quick sanity check that the PCA components actually relate
# to the target.
print("=" * 60)
print("4. CORRELATION ANALYSIS")
print("=" * 60)

v_cols = [f"V{i}" for i in range(1, 29)]
feature_cols = v_cols + ["Amount"]
corr_with_class = (
    df[feature_cols + ["Class"]].corr()["Class"].drop("Class").sort_values()
)

print("  Features most negatively correlated with Class (fraud):")
for feat, corr in corr_with_class.head(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

print("\n  Features most positively correlated with Class (fraud):")
for feat, corr in corr_with_class.tail(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

# Horizontal bar — green = positive (more of this feature → more likely fraud),
# red = negative (more of this feature → more likely legit).
fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values,
        color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Pearson Correlation with Class", fontsize=12)
ax.set_title("Feature Correlation with Fraud Label", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.8)
fig.tight_layout()
save_fig(fig, "03_feature_correlation_with_class.png")


## Phase 4 — Preprocessing utilities (10-fold splits, no SMOTE)

In [ ]:
# ── Feature / target separation ──
# X = 29 features (V1–V28 + Amount). y = binary Class label.
# We drop Class from X but keep Amount — Amount is a real feature and will be
# StandardScaled along with the V components.
print("=" * 60)
print("5. PREPROCESSING PIPELINE")
print("=" * 60)

X = df.drop(columns=["Class"])
y = df["Class"]

print(f"  Feature matrix X: {X.shape}")
print(f"  Target vector y:  {y.shape}  "
      f"(fraud={y.sum():,}, legit={len(y) - y.sum():,})")


In [ ]:
def create_kfold_splits(X, y, n_folds=N_FOLDS, random_state=RANDOM_STATE):
    """Generate 10-fold shuffled (train_idx, test_idx) pairs.

    We use plain KFold (not StratifiedKFold) because the dataset is already
    50/50 — any random shuffle preserves that ratio in expectation, and
    stratification adds compute cost without changing the result.

    shuffle=True with a fixed seed gives reproducible folds.
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    return list(kf.split(X, y))


def scale_split(X, y, train_idx, test_idx):
    """Fit StandardScaler on the training fold and transform both train and test.

    Critical: the scaler is fit ONLY on training data. Fitting on the full
    dataset would leak test-set statistics into the training pipeline.
    No SMOTE is applied — the dataset is already balanced.

    Returns (X_train_scaled, y_train, X_test_scaled, y_test, scaler).
    """
    feature_names = X.columns.tolist()

    X_train_raw = X.iloc[train_idx]
    X_test_raw = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_raw),
        columns=feature_names, index=X_train_raw.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test_raw),
        columns=feature_names, index=X_test_raw.index,
    )

    return X_train_scaled, y_train, X_test_scaled, y_test, scaler


# Build the splits once and reuse across all three algorithms so every model is
# evaluated on identical train/test partitions (fair comparison).
splits = create_kfold_splits(X, y)

print("\n" + "=" * 60)
print("6. 10-FOLD KFOLD CROSS VALIDATION SETUP")
print("=" * 60)
print(f"  Number of folds: {N_FOLDS}")
print(f"  Shuffle: True (seed={RANDOM_STATE})  |  No stratification, no SMOTE\n")

print(f"  {'Fold':<6} {'Train Size':>12} {'Train Fraud':>13} "
      f"{'Test Size':>11} {'Test Fraud':>12} {'Test Fraud %':>14}")
print(f"  {'-' * 72}")
for i, (train_idx, test_idx) in enumerate(splits, start=1):
    train_fraud = y.iloc[train_idx].sum()
    test_fraud = y.iloc[test_idx].sum()
    test_pct = test_fraud / len(test_idx) * 100
    print(
        f"  {i:<6} {len(train_idx):>12,} {train_fraud:>13,} "
        f"{len(test_idx):>11,} {test_fraud:>12,} {test_pct:>13.2f}%"
    )


## Phase 5 — Evaluation helpers (shared by all models)

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    """Standard 5-metric suite used for all three algorithms.

    Precision: of predicted-fraud, how many really are fraud (false-alarm cost).
    Recall:    of actual fraud, how many we caught (missed-fraud cost).
    F1:        harmonic mean of precision and recall.
    MCC:       balanced single-number score using all 4 confusion-matrix cells.
    AUC-ROC:   threshold-independent ranking quality.
    """
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "mcc":       matthews_corrcoef(y_true, y_pred),
        "auc_roc":   roc_auc_score(y_true, y_prob),
    }


def print_classification_report(y_true, y_pred):
    """Pretty per-class precision/recall/F1 from sklearn."""
    print(classification_report(
        y_true, y_pred,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0,
    ))


def print_summary_table(results_df, model_name):
    """Print mean ± std of every metric across all folds.

    Mean = expected performance; std = stability across random data partitions
    (low std = consistent, robust model).
    """
    print(f"\n  -- {model_name} Summary Across All Folds --")
    print(f"  {'Metric':<12} {'Mean':>10} {'Std':>10}")
    print(f"  {'-' * 32}")
    for metric in ["precision", "recall", "f1", "mcc", "auc_roc"]:
        mean_val = results_df[metric].mean()
        std_val = results_df[metric].std()
        print(f"  {metric:<12} {mean_val:>10.4f} {std_val:>10.4f}")


In [ ]:
# ── Plotting helpers (model-agnostic) ──
# These take callables (or precomputed predictions) rather than trained models,
# so they can be reused across LR, RF, and FFNN with no changes.

def plot_confusion_matrices(splits, X_features, y, train_and_predict_fn,
                            model_name, filename):
    """Grid of per-fold confusion matrices. train_and_predict_fn(train_idx, test_idx)
    must return y_pred for that fold (already scaled internally)."""
    fig, axes = plt.subplots(1, len(splits), figsize=(4 * len(splits), 4))
    if len(splits) == 1:
        axes = [axes]

    for i, (train_idx, test_idx) in enumerate(splits, start=1):
        y_pred = train_and_predict_fn(train_idx, test_idx)
        y_test = y.iloc[test_idx]

        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i - 1],
                    xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"],
                    cbar=False)
        axes[i - 1].set_title(f"Fold {i}", fontsize=11, fontweight="bold")
        axes[i - 1].set_ylabel("Actual"); axes[i - 1].set_xlabel("Predicted")

    fig.suptitle(f"{model_name} - Confusion Matrices", fontsize=16,
                 fontweight="bold", y=1.02)
    fig.tight_layout()
    save_fig(fig, filename)


def plot_roc_curves(splits, X_features, y, train_and_predict_proba_fn,
                    model_name, filename):
    """All ROC curves overlaid — visually shows AUC consistency across folds."""
    fig, ax = plt.subplots(figsize=(8, 6))

    for i, (train_idx, test_idx) in enumerate(splits, start=1):
        y_prob = train_and_predict_proba_fn(train_idx, test_idx)
        y_test = y.iloc[test_idx]

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"Fold {i} (AUC={auc_val:.4f})", alpha=0.7)

    # Diagonal = random classifier baseline.
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Classifier")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"{model_name} - ROC Curves", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    save_fig(fig, filename)


def plot_metrics_bars(results_df, splits, model_name, filename):
    """Grouped bar chart: 5 metrics × N folds. Quick visual stability check."""
    fig, ax = plt.subplots(figsize=(14, 5))

    metrics_to_plot = ["precision", "recall", "f1", "mcc", "auc_roc"]
    x = np.arange(len(splits))
    width = 0.15
    colors_metrics = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6", "#f39c12"]

    for i, metric in enumerate(metrics_to_plot):
        ax.bar(x + i * width, results_df[metric].values, width,
               label=metric.upper().replace("_", "-"), color=colors_metrics[i])

    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"{model_name} - Metrics by Fold", fontsize=14, fontweight="bold")
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels([f"F{i+1}" for i in range(len(splits))])
    ax.legend(loc="lower right"); ax.set_ylim(0, 1.05)
    fig.tight_layout()
    save_fig(fig, filename)


def plot_feature_coefficients(coefs, feature_names, model_name, fold_label,
                              filename, top_n=15):
    """LR-only: top-N features by |coefficient| with sign-coloured bars."""
    fig, ax = plt.subplots(figsize=(12, 6))
    sorted_idx = np.argsort(np.abs(coefs))[::-1]

    ax.barh(range(top_n), coefs[sorted_idx[:top_n]],
            color=["#e74c3c" if c < 0 else "#2ecc71"
                   for c in coefs[sorted_idx[:top_n]]],
            edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx[:top_n]])
    ax.set_xlabel("Coefficient Value", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Feature Coefficients ({fold_label})",
                 fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, filename)


def plot_feature_importances(importances, feature_names, model_name, fold_label,
                             filename, top_n=15):
    """RF-only: top-N features by Gini importance. All bars are non-negative."""
    fig, ax = plt.subplots(figsize=(12, 6))
    sorted_idx = np.argsort(importances)[::-1][:top_n]

    ax.barh(range(top_n), importances[sorted_idx],
            color="#3498db", edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx])
    ax.set_xlabel("Gini Importance", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Feature Importances ({fold_label})",
                 fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, filename)


## Phase 6 — Algorithm 1: Logistic Regression

L2-regularised linear baseline. Because the dataset is balanced, no `class_weight`
adjustment is needed and SMOTE is skipped.

Objective: $Z_1 = \min_{w,c} -\frac{1}{n}\sum_i [y_i \log\sigma(w^\top x_i + c) + (1-y_i)\log(1-\sigma(w^\top x_i + c))] + \lambda\|w\|_2^2$


In [ ]:
def _build_lr():
    """Factory: fresh L2-penalised LogisticRegression with consistent hyperparameters.

    lbfgs is a quasi-Newton solver that handles L2 efficiently on dense data.
    max_iter=1000 is generous to avoid early-stopping convergence warnings.
    """
    return LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=RANDOM_STATE,
    )


print("=" * 60)
print("7. LOGISTIC REGRESSION - TRAINING & EVALUATION")
print("=" * 60)

# Cache feature names for the coefficient plot at the end of the LR section.
feature_names = X.columns.tolist()

lr_results = []        # per-fold metric dicts
lr_last_model = None   # keep the last fold's fitted model for coefficient plotting

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    print(f"\n  -- Fold {split_num} --")

    # Scale features (no SMOTE).
    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    print(f"    Train: {len(X_train):,}  |  Test: {len(X_test):,}")

    # Fit -> predict -> metrics.
    model = _build_lr()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    lr_results.append(metrics)

    print(f"    Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}  |  "
          f"F1: {metrics['f1']:.4f}  |  MCC: {metrics['mcc']:.4f}  |  "
          f"AUC-ROC: {metrics['auc_roc']:.4f}")

    lr_last_model = model

# Persist results to disk for the report and downstream comparison.
lr_df = pd.DataFrame(lr_results)
lr_df.to_csv(os.path.join(RESULTS_DIR, "lr_results.csv"), index=False)
print_summary_table(lr_df, "Logistic Regression")


In [ ]:
# ── LR visualisations ──
# We re-train per fold inside the plotting helpers. This is wasteful for LR
# (cheap model) but keeps the helper signatures uniform across all 3 algorithms.
# The RF and FFNN sections will use a caching trick to avoid re-training.
print("=" * 60)
print("8. LOGISTIC REGRESSION - VISUALISATIONS")
print("=" * 60)

def _lr_predict(train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_split(X, y, train_idx, test_idx)
    m = _build_lr(); m.fit(X_tr, y_tr)
    return m.predict(X_te)

def _lr_predict_proba(train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_split(X, y, train_idx, test_idx)
    m = _build_lr(); m.fit(X_tr, y_tr)
    return m.predict_proba(X_te)[:, 1]

plot_confusion_matrices(splits, X, y, _lr_predict,
                        "Logistic Regression", "11_lr_confusion_matrices.png")
plot_roc_curves(splits, X, y, _lr_predict_proba,
                "Logistic Regression", "12_lr_roc_curves.png")
plot_metrics_bars(lr_df, splits, "Logistic Regression",
                  "13_lr_metrics_by_fold.png")
# Coefficients from the last fold's fitted model — sign + magnitude reveal
# which PCA components drive the linear fraud decision.
plot_feature_coefficients(lr_last_model.coef_[0], feature_names,
                          "Logistic Regression", f"Fold {len(splits)}",
                          "14_lr_feature_coefficients.png")

print("\nLogistic Regression complete.")


## Phase 7 — Algorithm 2: Random Forest

Ensemble of 200 decision trees, each trained on a bootstrap sample. RF handles
non-linear interactions natively and gives us Gini feature importances for
interpretability.

> **Heads up**: RF is the slowest algorithm in this notebook. 200 trees × 10 folds
> on ~511K training rows takes a few minutes even on a fast CPU. `n_jobs=-1`
> parallelises across all available cores.


In [ ]:
def _build_rf():
    """Factory: fresh RandomForestClassifier.

    n_estimators=200 is a sensible default — enough trees for stable predictions
    without absurd training time. We omit class_weight because the dataset is
    already balanced (using class_weight='balanced' on balanced data would
    actually distort the loss).
    """
    return RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


print("=" * 60)
print("9. RANDOM FOREST - TRAINING & EVALUATION")
print("=" * 60)

rf_results = []
rf_last_model = None
# Cache predictions per fold so the visualisation cell doesn't have to retrain.
rf_fold_predictions = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    print(f"\n  -- Fold {split_num} --")

    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    print(f"    Train: {len(X_train):,}  |  Test: {len(X_test):,}")

    model = _build_rf()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    rf_results.append(metrics)
    rf_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}

    print(f"    Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}  |  "
          f"F1: {metrics['f1']:.4f}  |  MCC: {metrics['mcc']:.4f}  |  "
          f"AUC-ROC: {metrics['auc_roc']:.4f}")

    rf_last_model = model

rf_df = pd.DataFrame(rf_results)
rf_df.to_csv(os.path.join(RESULTS_DIR, "rf_results.csv"), index=False)
print_summary_table(rf_df, "Random Forest")


In [ ]:
# ── RF visualisations ──
# Reuse cached predictions via a closure-based predictor. The plot helpers don't
# know we're cheating — they just see a function f(train_idx, test_idx) that
# returns the right predictions.
print("=" * 60)
print("10. RANDOM FOREST - VISUALISATIONS")
print("=" * 60)

def _make_cached_predictor(fold_predictions, key, n_folds):
    """Return a function that yields cached predictions in fold order."""
    preds = [fold_predictions[i + 1][key] for i in range(n_folds)]
    call_idx = [0]
    def predictor(train_idx, test_idx):
        result = preds[call_idx[0]]
        call_idx[0] += 1
        return result
    return predictor

plot_confusion_matrices(
    splits, X, y,
    _make_cached_predictor(rf_fold_predictions, "y_pred", len(splits)),
    "Random Forest", "15_rf_confusion_matrices.png",
)
plot_roc_curves(
    splits, X, y,
    _make_cached_predictor(rf_fold_predictions, "y_prob", len(splits)),
    "Random Forest", "16_rf_roc_curves.png",
)
plot_metrics_bars(rf_df, splits, "Random Forest", "17_rf_metrics_by_fold.png")
# Feature importances from the last fold — Gini impurity reduction averaged
# across all 200 trees, normalised to sum to 1.
plot_feature_importances(rf_last_model.feature_importances_, feature_names,
                         "Random Forest", f"Fold {len(splits)}",
                         "18_rf_feature_importances.png")

print("\nRandom Forest complete.")


## Phase 8 — Algorithm 3: Feed-Forward Neural Network

Two hidden layers (64 → 32) with BatchNorm + Dropout, trained with Adam + BCE loss.
Architecture: `Input(29) → Dense(64, ReLU, BN, Drop) → Dense(32, ReLU, BN, Drop) → Dense(1, Sigmoid)`.

Same hyperparameters as the Dataset 3 notebook, enabling direct cross-dataset comparison.


In [ ]:
# ── FFNN hyperparameters ──
# Kept identical to Dataset 3 so cross-dataset comparison is meaningful.
HIDDEN1 = 64           # first hidden layer width
HIDDEN2 = 32           # second hidden layer width (compresses representation)
DROPOUT = 0.3          # zero-out probability — regularises against overfitting
EPOCHS = 50            # full passes through the training fold
BATCH_SIZE = 256       # mini-batch size for SGD
LEARNING_RATE = 1e-3   # Adam step size
WEIGHT_DECAY = 1e-4    # L2 regularisation strength applied inside Adam


class FraudDetectorNN(nn.Module):
    """Sequential FFNN: 29 -> 64 -> 32 -> 1 with BN + ReLU + Dropout.

    BatchNorm normalises each mini-batch's activations — stabilises training
    and acts as a mild regulariser. Dropout randomly zeros 30% of neurons each
    forward pass during training to discourage co-adaptation. Final Sigmoid
    maps the logit to a probability in [0, 1].
    """

    def __init__(self, input_dim, hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # squeeze(-1) drops the trailing singleton dim so the output shape
        # matches the target shape (batch_size,) for BCELoss.
        return self.network(x).squeeze(-1)


def _set_seeds():
    """Seed every RNG involved in training so per-fold runs are reproducible."""
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)


def _train_nn(X_train, y_train, input_dim, epochs=EPOCHS, batch_size=BATCH_SIZE,
              lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    """Train one FFNN from scratch on (X_train, y_train).

    Returns (trained_model, list_of_per_epoch_avg_losses) so we can plot
    convergence curves later.
    """
    _set_seeds()

    model = FraudDetectorNN(input_dim).to(DEVICE)
    criterion = nn.BCELoss()                                   # binary cross-entropy
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=weight_decay)

    # Convert pandas/numpy inputs to float32 tensors.
    X_arr = X_train.values if hasattr(X_train, "values") else np.array(X_train)
    y_arr = y_train.values if hasattr(y_train, "values") else np.array(y_train)
    X_tensor = torch.FloatTensor(X_arr)
    y_tensor = torch.FloatTensor(y_arr)

    # Seeded generator so the shuffled batch order is reproducible across runs.
    dataset = TensorDataset(X_tensor, y_tensor)
    g = torch.Generator().manual_seed(RANDOM_STATE)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=g)

    epoch_losses = []
    model.train()  # enables dropout + BatchNorm running-stats updates

    for epoch in range(epochs):
        running_loss = 0.0
        n_samples = 0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            # Standard PyTorch step: zero grads -> forward -> loss -> backward -> step.
            optimizer.zero_grad()
            y_hat = model(X_batch)
            loss = criterion(y_hat, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(X_batch)
            n_samples += len(X_batch)

        avg_loss = running_loss / n_samples
        epoch_losses.append(avg_loss)

        # Lightweight progress logging: first epoch + every 10th.
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"      Epoch {epoch + 1:>3}/{epochs}  |  Loss: {avg_loss:.6f}")

    return model, epoch_losses


def _predict(model, X_test):
    """Run inference on the test fold.

    model.eval() disables dropout and switches BatchNorm to use stored running
    statistics. torch.no_grad() skips autograd bookkeeping. Threshold at 0.5 to
    convert probabilities to hard labels.
    """
    model.eval()
    with torch.no_grad():
        X_arr = X_test.values if hasattr(X_test, "values") else np.array(X_test)
        X_tensor = torch.FloatTensor(X_arr).to(DEVICE)
        y_prob = model(X_tensor).cpu().numpy()
    y_pred = (y_prob >= 0.5).astype(int)
    return y_pred, y_prob


In [ ]:
# ── FFNN training & evaluation across all folds ──
# A fresh network is trained per fold so each fold is a truly independent
# evaluation (no leakage of weights from earlier folds).
print("=" * 60)
print("11. FEED-FORWARD NEURAL NETWORK - TRAINING & EVALUATION")
print("=" * 60)

input_dim = X.shape[1]

print(f"\n  Architecture: Input({input_dim}) -> Dense({HIDDEN1}, ReLU, BN, Drop) "
      f"-> Dense({HIDDEN2}, ReLU, BN, Drop) -> Dense(1, Sigmoid)")
print(f"  Optimiser:    Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"  Loss:         Binary Cross-Entropy")
print(f"  Epochs: {EPOCHS}  |  Batch Size: {BATCH_SIZE}  |  Dropout: {DROPOUT}")
print(f"  Device:       {DEVICE}")

nn_results = []
# Cache predictions and loss curves so visualisation doesn't retrain the network.
nn_fold_predictions = {}
all_epoch_losses = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    print(f"\n  -- Fold {split_num} --")

    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    print(f"    Train: {len(X_train):,}  |  Test: {len(X_test):,}")

    model, epoch_losses = _train_nn(X_train, y_train, input_dim)
    y_pred, y_prob = _predict(model, X_test)

    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    nn_results.append(metrics)

    nn_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    all_epoch_losses[split_num] = epoch_losses

    print(f"    Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}  |  "
          f"F1: {metrics['f1']:.4f}  |  MCC: {metrics['mcc']:.4f}  |  "
          f"AUC-ROC: {metrics['auc_roc']:.4f}")

nn_df = pd.DataFrame(nn_results)
nn_df.to_csv(os.path.join(RESULTS_DIR, "nn_results.csv"), index=False)
print_summary_table(nn_df, "Feed-Forward Neural Network")


In [ ]:
# ── FFNN visualisations ──
# Same caching trick as the RF section — predictions were stored during training.
print("=" * 60)
print("12. FEED-FORWARD NEURAL NETWORK - VISUALISATIONS")
print("=" * 60)

# Training loss curves: one line per fold. All folds should converge to similar
# loss values since 10-fold KFold gives roughly equivalent training set sizes.
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("tab10")
for split_num in sorted(all_epoch_losses.keys()):
    losses = all_epoch_losses[split_num]
    ax.plot(range(1, len(losses) + 1), losses,
            label=f"Fold {split_num}", color=cmap((split_num - 1) % 10), linewidth=1.3)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
ax.set_title("Feed-Forward Neural Network - Training Loss Curves",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8); ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "19_nn_training_loss.png")

# Reuse the cached-predictor helper from the RF section.
plot_confusion_matrices(
    splits, X, y,
    _make_cached_predictor(nn_fold_predictions, "y_pred", len(splits)),
    "Feed-Forward Neural Network", "20_nn_confusion_matrices.png",
)
plot_roc_curves(
    splits, X, y,
    _make_cached_predictor(nn_fold_predictions, "y_prob", len(splits)),
    "Feed-Forward Neural Network", "21_nn_roc_curves.png",
)
plot_metrics_bars(nn_df, splits, "Feed-Forward Neural Network",
                  "22_nn_metrics_by_fold.png")

print("\nFeed-Forward Neural Network complete.")


## Phase 9 — Cross-model summary

In [ ]:
# Combine the per-fold metric DataFrames into one mean-per-model table.
# This is the headline result: which algorithm wins on the balanced 2023 dataset?
summary_frames = {
    "Logistic Regression": lr_df,
    "Random Forest":       rf_df,
    "Feed-Forward NN":     nn_df,
}

metric_cols = ["precision", "recall", "f1", "mcc", "auc_roc"]
summary = pd.DataFrame({
    name: frame[metric_cols].mean()
    for name, frame in summary_frames.items()
}).T

print("Mean metrics across all 10 folds:")
print(summary.round(4))

# Persist the summary too — useful for the report.
summary.to_csv(os.path.join(RESULTS_DIR, "summary_mean_metrics.csv"))


## Phase 10 — Download artefacts (optional)

All plots and CSVs are written to `./results/` in the Colab session. To pull them
down locally:

```python
import shutil
from google.colab import files
shutil.make_archive("dataset2_results", "zip", "results")
files.download("dataset2_results.zip")
```
